# 자연어 처리 6일차 (1) — GPT 와 BERT · 마스크 언어 모델(MLM) · 다음 문장 예측(NSP) 직접 돌려 보기

교재 `자연어처리` 교재v1 **p.97~107** (Transformer 실습 · GPT · BERT · MLM 동작 과정 1~5 · NSP · contextual word embedding) 를 **사전 학습된 BERT 로 직접 확인한 노트**입니다. 함께 받은 BERT 논문(Devlin 외, 2018)의 숫자도 대조했습니다.

- `pipeline()` 추론(필기 `nlp_실습9`)은 [`pipeline_inference.ipynb`](./pipeline_inference.ipynb), 네이버 영화 리뷰 미세 조정(필기 `nlp_실습10_전이학습`)은 [`nsmc_finetune_bert.ipynb`](./nsmc_finetune_bert.ipynb) 에 있습니다.
- 교재에서 고친 부분은 **📝 고친 부분** 으로 적었습니다. **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 있습니다.

| | GPT (OpenAI, 2018) | BERT (Google, 2018) |
|---|---|---|
| Transformer 에서 쓰는 부분 | **디코더**만 | **인코더**만 |
| 사전 학습 과제 | 앞 단어들 → **다음 단어** 맞히기 | 빈칸 **[MASK]** 맞히기(MLM) + 다음 문장 맞히기(NSP) |
| 보는 방향 | 왼쪽 → 오른쪽 (한 방향) | 빈칸 **앞뒤 모두** (양방향) |
| 강점 | 문장 생성 (쓰기) | 문장 의미 추출 (읽고 이해하기) |

## 0. 설치 · 실행 방법

| 패키지 | 이 노트에서 쓰는 것 | 설치 |
|---|---|---|
| `transformers` | `AutoTokenizer` · `BertForMaskedLM` · `BertForNextSentencePrediction` · `pipeline('fill-mask')` | Colab 기본 포함 · 로컬은 `pip install transformers` |
| `torch` | 모델 실행 | Colab 기본 포함 · 로컬은 `pip install torch` |

**모델** : 첫 실행 때 Hugging Face 에서 자동으로 받는다 → **인터넷 필요**
- `klue/bert-base` (한국어 BERT, 약 445MB) — 다음 노트의 미세 조정에도 그대로 쓴다
- `google-bert/bert-base-uncased` (영어 BERT, 약 440MB) — 교재 p.100 의 영어 예문 · NSP

**실행 순서** : 맨 위부터 **[런타임] → [모두 실행]**. 학습하지 않고 예측만 하므로 GPU 없이 CPU 로 1~2분(다운로드 시간 제외).

In [1]:
# ① 설치 (로컬에서 처음 한 번 — Colab 은 기본 포함)
# !pip install -q transformers torch

In [2]:
import warnings, os
warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'      # 다운로드 진행 막대 숨기기
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()                      # 'LOAD REPORT' 같은 안내 줄 숨기기 (아래 셀에서 직접 확인)
hf_logging.disable_progress_bar()
from huggingface_hub.utils import logging as hub_logging
hub_logging.set_verbosity_error()                     # 'unauthenticated requests' 안내 숨기기 (로그인 없이도 공개 모델은 받아진다)

In [3]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, BertForMaskedLM, BertForNextSentencePrediction, pipeline
_ = torch.set_grad_enabled(False)             # 예측만 — 기울기 계산 끄기

## 1. GPT 와 BERT 의 크기 (교재 p.98~99 · BERT 논문)

In [4]:
ko_tok = AutoTokenizer.from_pretrained('klue/bert-base')
ko_mlm = BertForMaskedLM.from_pretrained('klue/bert-base')
en_tok = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased')

n_params = sum(p.numel() for p in ko_mlm.bert.parameters())
c = ko_mlm.config
print(f'klue/bert-base 본체 가중치 : {n_params:,}개 (약 {n_params / 1e6:.0f}M)')
print(f'층 {c.num_hidden_layers} · hidden {c.hidden_size} · head {c.num_attention_heads} · 사전 {c.vocab_size:,} · 최대 길이 {c.max_position_embeddings}')

klue/bert-base 본체 가중치 : 110,026,752개 (약 110M)
층 12 · hidden 768 · head 12 · 사전 32,000 · 최대 길이 512


| 모델 | 가중치 | 비고 |
|---|---|---|
| GPT-1 | 1억 1,700만 (117M) | 교재 p.98 ✓ |
| GPT-2 | 15억 4,200만 (1.5B, 가장 큰 모델) | 교재 p.98 ✓ |
| BERT-base / large | **110M / 340M** (논문 표기) | 교재 '1억1천만 ~ 3억4천5백만' — large 는 논문에 340M |
| BERT 사전 학습 데이터 | BooksCorpus 8억 + 영어 위키백과 25억 단어 = **33억 단어** | 교재 p.99 ✓ |

> **📝 고친 부분 (교재 p.99)** BERT-large 의 가중치 수는 논문(Devlin 외, 2018) 표기로 **340M**. 실제로 세면 약 3억 3,500만 ~ 3억 4,000만 개라 '3억 4,500만' 은 조금 크다. 위 klue/bert-base 처럼 base 크기는 약 1억 1,000만 개.

## 2. 입력 형식 — `[CLS] 문장 A [SEP] 문장 B [SEP]` (교재 p.100~102)

In [5]:
enc = en_tok('my dog is cute', 'he likes playing')
print('토큰     :', en_tok.convert_ids_to_tokens(enc['input_ids']))
print('번호     :', enc['input_ids'])
print('문장 구분 :', enc['token_type_ids'], '← 0 = 문장 A, 1 = 문장 B (Segment 임베딩)')

print('\n특수 토큰 번호 — 영어 BERT :', {t: en_tok.convert_tokens_to_ids(t) for t in ['[PAD]', '[CLS]', '[SEP]', '[MASK]']})
print('특수 토큰 번호 — klue BERT :', {t: ko_tok.convert_tokens_to_ids(t) for t in ['[PAD]', '[CLS]', '[SEP]', '[MASK]']})

토큰     : ['[CLS]', 'my', 'dog', 'is', 'cute', '[SEP]', 'he', 'likes', 'playing', '[SEP]']
번호     : [101, 2026, 3899, 2003, 10140, 102, 2002, 7777, 2652, 102]
문장 구분 : [0, 0, 0, 0, 0, 0, 1, 1, 1, 1] ← 0 = 문장 A, 1 = 문장 B (Segment 임베딩)

특수 토큰 번호 — 영어 BERT : {'[PAD]': 0, '[CLS]': 101, '[SEP]': 102, '[MASK]': 103}
특수 토큰 번호 — klue BERT : {'[PAD]': 0, '[CLS]': 2, '[SEP]': 3, '[MASK]': 4}


In [6]:
emb = ko_mlm.bert.embeddings
print('Token    임베딩 :', tuple(emb.word_embeddings.weight.shape), '(사전 크기, 768)')
print('Segment  임베딩 :', tuple(emb.token_type_embeddings.weight.shape), '(문장 A/B 2가지, 768)')
print('Position 임베딩 :', tuple(emb.position_embeddings.weight.shape), '(최대 길이 512, 768) — 학습하는 위치 임베딩')

x = ko_tok('비행기를 타기 위해 공항에 간다', return_tensors='pt')
ids, L = x['input_ids'], x['input_ids'].shape[1]
manual = emb.LayerNorm(emb.word_embeddings(ids) + emb.token_type_embeddings(x['token_type_ids']) + emb.position_embeddings(torch.arange(L)[None]))
print('\n세 임베딩을 더해 LayerNorm == BERT 임베딩 층 ?', torch.allclose(manual, emb(input_ids=ids, token_type_ids=x['token_type_ids']), atol=1e-5))

Token    임베딩 : (32000, 768) (사전 크기, 768)
Segment  임베딩 : (2, 768) (문장 A/B 2가지, 768)
Position 임베딩 : (512, 768) (최대 길이 512, 768) — 학습하는 위치 임베딩

세 임베딩을 더해 LayerNorm == BERT 임베딩 층 ? True


> **📝 고친 부분 (교재 p.101)** 표의 토큰 번호 `CLS 2 · SEP 3 · MASK 4` 는 **klue/bert-base 사전과 같다**(위 결과). 영어 `bert-base-uncased` 는 `[CLS] 101 · [SEP] 102 · [MASK] 103` — 번호는 모델(사전)마다 다르다. `my 1000 · is 1002` 같은 번호는 설명용. 세 임베딩을 더한 뒤 **LayerNorm** 까지 거쳐야 BERT 임베딩 층의 출력과 같아진다.

## 3. 마스크 언어 모델(MLM) — 빈칸 맞히기 (교재 p.99 · 101 · 105)

In [7]:
fill_ko = pipeline('fill-mask', model=ko_mlm, tokenizer=ko_tok)
sent = '비행기를 타기 위해 공항을 가는데 [MASK] 너무 막혀서 결국 비행기를 놓쳤다.'
for r in fill_ko(sent, top_k=5):
    print(f"{r['score']:.3f}  {r['token_str']}")

0.607  길이
0.300  차가
0.048  ,
0.007  줄이
0.004  ##도


In [8]:
# 교재 p.105 : [MASK] 자리의 최종 벡터(768) → Dense → 사전 전체(32,000개)에 대한 확률
x = ko_tok(sent, return_tensors='pt')
logits = ko_mlm(**x).logits                                   # (1, 길이, 32000)
pos = (x['input_ids'][0] == ko_tok.mask_token_id).nonzero().item()
probs = F.softmax(logits[0, pos], dim=-1)
print('출력 units = 사전 크기 :', logits.shape[-1], '| 확률 합 :', round(probs.sum().item(), 4))
top = probs.topk(3)
print('상위 3개 :', [(ko_tok.convert_ids_to_tokens(int(i)), round(p.item(), 3)) for p, i in zip(top.values, top.indices)])

출력 units = 사전 크기 : 32000 | 확률 합 : 1.0


상위 3개 : [('길이', 0.607), ('차가', 0.3), (',', 0.048)]


In [9]:
# 양방향 : [MASK] 뒤쪽 문맥을 바꾸면 답이 바뀐다
for s in ['오늘 [MASK]가 너무 막혀서 결국 비행기를 놓쳤다.',
          '오늘 [MASK]가 너무 아파서 결국 병원에 갔다.',
          '오늘 [MASK]가 너무 고파서 결국 라면을 끓였다.']:
    print(s, '→', [(r['token_str'], round(r['score'], 2)) for r in fill_ko(s, top_k=3)])

오늘 [MASK]가 너무 막혀서 결국 비행기를 놓쳤다. → [('비행기', 0.39), ('##은', 0.06), ('##도', 0.04)]


오늘 [MASK]가 너무 아파서 결국 병원에 갔다. → [('아침', 0.06), ('하루', 0.06), ('머리', 0.04)]


오늘 [MASK]가 너무 고파서 결국 라면을 끓였다. → [('배', 0.85), ('##은', 0.06), ('##도', 0.02)]


- 교재 예문의 1위는 `차가`가 아니라 `길이`(0.61), `차가` 는 2위(0.30) — 둘 다 말이 된다. 정답이 하나로 정해진 게 아니라 **확률 분포**다.
- 아래 세 문장은 `[MASK]` 앞이 모두 `오늘` 로 같은데 **뒤 문맥**(막혀서 · 아파서 · 고파서)에 따라 1위 답이 달라진다 — 왼쪽만 보는 GPT 식 모델은 할 수 없는 일이다(교재 p.99 '양방향'). `[MASK]` 바로 뒤에 조사(`가`)를 붙이지 않으면 `##은` 같은 조각이 1위로 나온다 — 빈칸이 단어인지 조사인지도 문맥으로 정한다. 첫 문장의 1위 `비행기`(0.39)는 뒤에 나오는 '비행기를 놓쳤다' 에 끌린 답이다 — 앞뒤를 다 보지만 늘 사람처럼 맞히지는 않는다(그 아래로 `기차` · `KTX` · `버스` · `고속도로` 처럼 '막히는' 교통 관련 단어가 이어진다).

### 3-1. 학습 때는 어떻게 가리나 — 15% 규칙 (BERT 논문 3.1)

In [10]:
from transformers import DataCollatorForLanguageModeling
collator = DataCollatorForLanguageModeling(tokenizer=ko_tok, mlm_probability=0.15)
torch.manual_seed(0)
texts = ['비행기를 타기 위해 공항을 가는데 차가 너무 막혀서 결국 비행기를 놓쳤다.'] * 2000
batch = collator([ko_tok(t) for t in texts])
orig = torch.tensor([ko_tok(t)['input_ids'] for t in texts])
chosen = batch['labels'] != -100                         # 맞혀야 할(가려진) 자리
special = (orig == ko_tok.cls_token_id) | (orig == ko_tok.sep_token_id)
print(f'가려진 비율 : {chosen.sum() / (~special).sum():.1%}  (특수 토큰 제외)')
sel_in = batch['input_ids'][chosen]; sel_orig = orig[chosen]
print(f'그중 [MASK] 로 바뀜 : {(sel_in == ko_tok.mask_token_id).float().mean():.1%}')
print(f'다른 단어로 바뀜     : {((sel_in != ko_tok.mask_token_id) & (sel_in != sel_orig)).float().mean():.1%}')
print(f'그대로 둠            : {(sel_in == sel_orig).float().mean():.1%}')
print('\n예 :', ko_tok.decode(batch['input_ids'][0]))

가려진 비율 : 14.9%  (특수 토큰 제외)
그중 [MASK] 로 바뀜 : 80.5%
다른 단어로 바뀜     : 9.6%
그대로 둠            : 9.9%

예 : [CLS] 비행기 [MASK] [MASK]기 위해 공항을 가는데 차가 너무 [MASK]서 결국 비행기를 놓쳤다. [SEP]


> **📝 고친 부분 (교재 p.101 'text 입력 → 토큰화 + Masking')** 교재 예시는 `dog` 한 단어를 `[MASK]` 로 바꾸지만, 실제 BERT 사전 학습은 **토큰의 15%** 를 고르고 그중 **80% 는 [MASK] · 10% 는 무작위 단어 · 10% 는 그대로** 둔다(논문 3.1). 미세 조정 · 실제 사용 때는 [MASK] 가 나오지 않으므로 그 차이를 줄이려는 장치다.

## 4. 다음 문장 예측(NSP) (교재 p.106)

In [11]:
nsp = BertForNextSentencePrediction.from_pretrained('google-bert/bert-base-uncased')
pairs = [('my dog is cute.', 'he likes playing in the park.'),
         ('my dog is cute.', 'the stock market fell sharply today.'),
         ('i went to the airport.', 'my flight was delayed by two hours.'),
         ('i went to the airport.', 'penguins live in antarctica.')]
for a, b in pairs:
    x = en_tok(a, b, return_tensors='pt')
    p = F.softmax(nsp(**x).logits, dim=-1)[0]             # [CLS] 벡터 → 2개 분류
    print(f'IsNext {p[0]:.3f} | NotNext {p[1]:.3f}  :  {a} / {b}')

IsNext 1.000 | NotNext 0.000  :  my dog is cute. / he likes playing in the park.
IsNext 0.003 | NotNext 0.997  :  my dog is cute. / the stock market fell sharply today.
IsNext 1.000 | NotNext 0.000  :  i went to the airport. / my flight was delayed by two hours.
IsNext 0.015 | NotNext 0.985  :  i went to the airport. / penguins live in antarctica.


`[CLS]` 자리의 최종 벡터 하나로 **이어지는 문장(IsNext)인지** 를 고른다. 학습 때는 50% 는 실제 다음 문장, 50% 는 무작위 문장을 붙였다(교재 p.106 ✓). 이 `[CLS]` 벡터가 다음 노트의 **문장 분류(감성 분석)** 에도 그대로 쓰인다.

> 참고 : 이후 RoBERTa(2019) 는 NSP 를 빼도 성능이 떨어지지 않는다고 보고했다 — BERT 의 핵심은 MLM 이다.

## 5. contextual word embedding — 같은 '배' 다른 벡터 (교재 p.107)

In [12]:
def word_vec(sentence, word):
    x = ko_tok(sentence, return_tensors='pt')
    toks = ko_tok.convert_ids_to_tokens(x['input_ids'][0])
    i = toks.index(word)
    return ko_mlm.bert(**x).last_hidden_state[0, i]       # 마지막 층의 그 단어 벡터 (768)

s = {'과일 1': '배 를 깎아 먹었다', '과일 2': '시원한 배 한 조각을 먹었다',
     '바다 1': '배 를 타고 섬에 갔다', '바다 2': '큰 배 가 항구에 들어왔다'}
for k, v in s.items():
    print(k, ko_tok.tokenize(v))
vec = {k: word_vec(v, '배') for k, v in s.items()}
names = list(vec)
print('\n코사인 유사도 (같은 단어 \'배\')')
for i in range(4):
    print(f'{names[i]:5s}', ' '.join(f'{F.cosine_similarity(vec[names[i]], vec[names[j]], dim=0):.3f}' for j in range(4)))

과일 1 ['배', '를', '깎', '##아', '먹', '##었', '##다']
과일 2 ['시원', '##한', '배', '한', '조각', '##을', '먹', '##었', '##다']
바다 1 ['배', '를', '타고', '섬', '##에', '갔', '##다']
바다 2 ['큰', '배', '가', '항구', '##에', '들어왔', '##다']



코사인 유사도 (같은 단어 '배')
과일 1  1.000 0.687 0.656 0.597
과일 2  0.687 1.000 0.462 0.507
바다 1  0.656 0.462 1.000 0.827
바다 2  0.597 0.507 0.827 1.000


입력 임베딩(사전의 한 행)은 넷 모두 **같지만**, 12개 층의 self-attention 을 지난 벡터는 서로 달라진다. 이번 결과에서 '타는 배' 끼리(0.83)와 '먹는 배' 끼리(0.69)가 **뜻이 다른 쌍(0.46~0.66)보다 대체로 높다** — 문장 전체를 참고한 **문맥 반영 벡터**라서다(차이가 아주 깔끔하지는 않다 — 사전 학습만 한 BERT 의 한계).

## 정리

| 항목 | 내용 |
|---|---|
| 입력 | `[CLS] 문장 A [SEP] 문장 B [SEP]` · Token + Segment + Position 임베딩 → LayerNorm |
| MLM | 15% 선택 → 80% `[MASK]` · 10% 무작위 · 10% 그대로 → 사전 전체 softmax |
| NSP | `[CLS]` 벡터 → IsNext / NotNext |
| 출력 | 토큰마다 768칸 문맥 반영 벡터 (`last_hidden_state`) |
| 다음 | 이 사전 학습 가중치를 출발점으로 감성 분류 학습 → [`nsmc_finetune_bert.ipynb`](./nsmc_finetune_bert.ipynb) |